<a href="https://colab.research.google.com/github/brucenguyen0302-code/last-mile-route-duration/blob/main/notebooks/AT1_II_route_duration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting Last-Mile Delivery Route Duration

**42172 Introduction to Artificial Intelligence — AT1, Example II (regression)**

Delivery routes are planned by software, but drivers often change the planned stop order, so the time a route
really takes can differ a lot from the plan. This notebook builds regression models that predict the **actual
duration of a delivery route (in minutes)** using only information that is known before the route starts.

**Data:** Konovalenko, A., Hvattum, L. M., & Iversen, K. A. H. (2024). *Last-mile delivery route deviations
dataset: Planned vs. actual routes* [Data set]. Mendeley Data. https://doi.org/10.17632/kkwgfvmtxn.1

**Notebook structure**
1. Setup
2. Load the dataset
3. First look at the data
4. Check the dataset against its description
5. Build one row per route
6. Clean the route table
7. Explore the route table
8. Chronological train, validation and test split
9. Baseline models
10. Model development and tuning
11. Final evaluation on the test set
12. Statistical comparison
13. Figures
14. Predicting unseen routes

## 1. Setup

Mount Google Drive, fix one random seed so every run gives the same results, and create the project folders
for data, results and figures.

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

import os, glob, random, sys
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = '/content/gdrive/MyDrive/IntroToAI/AT1_II'
DATA_DIR = os.path.join(BASE, 'data')
RESULTS_DIR = os.path.join(BASE, 'results')
FIG_DIR = os.path.join(BASE, 'figures')
for folder in (DATA_DIR, RESULTS_DIR, FIG_DIR):
    os.makedirs(folder, exist_ok=True)

print('Python :', sys.version.split()[0])
print('pandas :', pd.__version__)
print('numpy  :', np.__version__)

Mounted at /content/gdrive
Python : 3.13.15
pandas : 2.2.3
numpy  : 2.1.3


## 2. Load the dataset

The data file is `routes_performance.xlsx`. The notebook searches Google Drive for it by name instead of using a
fixed path. Reading a 20 MB Excel file is slow, so the first run saves a fast copy (a pickle file) in the data
folder and later runs load that copy instead.

In [2]:
DATA_FILE = 'routes_performance.xlsx'
CACHE_FILE = os.path.join(DATA_DIR, 'routes_performance_raw.pkl')

if os.path.exists(CACHE_FILE):
    raw = pd.read_pickle(CACHE_FILE)
    print('Loaded from cache:', CACHE_FILE)
else:
    matches = glob.glob(f'/content/gdrive/MyDrive/**/{DATA_FILE}', recursive=True)
    assert matches, f'{DATA_FILE} not found anywhere in Google Drive'
    DATA_PATH = matches[0]
    print('Reading:', DATA_PATH)
    raw = pd.read_excel(DATA_PATH, sheet_name=0)
    raw.to_pickle(CACHE_FILE)
    print('Saved cache:', CACHE_FILE)

print(f'Rows    : {len(raw):,}')
print(f'Columns : {raw.shape[1]}')

Loaded from cache: /content/gdrive/MyDrive/IntroToAI/AT1_II/data/routes_performance_raw.pkl
Rows    : 249,231
Columns : 16


## 3. First look at the data

Each row is one visit to a stop on a route. This cell shows the first rows, counts missing values, and lists
each column's type, number of unique values and range.

In [3]:
display(raw.head(10))

print('Missing values in the whole table:', int(raw.isna().sum().sum()))

summary = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'unique values': raw.nunique(),
    'min': raw.min(numeric_only=True),
    'max': raw.max(numeric_only=True),
}).reindex(raw.columns)
pd.set_option('display.float_format', '{:,.2f}'.format)
display(summary)

,Route ID,Driver ID,Stop ID,Address ID,Week ID,Country,Day of Week,IndexP,IndexA,Arrived Time,Earliest Time,Latest Time,DistanceP,DistanceA,Depot,Delivery
0,0,0,0,0,0,1,Monday,0,0,42.275,0.0,360.0,0.000000,0.000000,1,0
1,0,0,1,1,0,1,Tuesday,1,4,332.788,240.0,480.0,16.329053,16.329053,0,1
2,0,0,2,2,0,1,Tuesday,2,5,332.956,120.0,540.0,0.373110,0.373110,0,1
3,0,0,3,3,0,1,Monday,3,2,244.994,60.0,540.0,2.491915,0.000000,0,1
4,0,0,4,3,0,1,Monday,4,1,244.855,60.0,540.0,0.000000,13.944962,0,1
5,0,0,0,0,0,1,Monday,5,3,272.121,0.0,360.0,13.944962,13.944962,1,0
6,0,0,1,1,0,1,Tuesday,6,6,373.553,240.0,480.0,16.329053,0.373110,0,1
7,1,1,0,0,0,1,Monday,0,0,64.855,0.0,360.0,0.000000,0.000000,1,0
8,1,1,5,4,0,1,Monday,1,2,75.520,120.0,540.0,16.767937,11.916344,0,1
9,1,1,6,5,0,1,Tuesday,2,5,324.371,120.0,540.0,0.080666,0.725253,0,1


Missing values in the whole table: 0


,dtype,unique values,min,max
Route ID,int64,19647,0.00,"19,646.00"
Driver ID,int64,400,0.00,399.00
Stop ID,int64,13125,0.00,"13,124.00"
Address ID,int64,10864,0.00,"10,863.00"
Week ID,int64,32,0.00,31.00
Country,int64,2,0.00,1.00
Day of Week,object,7,NaN,NaN
IndexP,int64,36,0.00,35.00
IndexA,int64,36,0.00,35.00
Arrived Time,float64,197396,0.00,"12,814,321.85"


## 4. Check the dataset against its description

The assertions below stop the notebook if the file is not the one expected, so any later change to the data is
caught straight away.

*Note:* the data paper reports 19,497 routes, but this file contains 19,647.

In [4]:
n_routes = raw['Route ID'].nunique()
n_drivers = raw['Driver ID'].nunique()
n_weeks = raw['Week ID'].nunique()

print(f'Routes    : {n_routes:,}')
print(f'Drivers   : {n_drivers}')
print(f"Weeks     : {n_weeks} (Week ID {raw['Week ID'].min()} to {raw['Week ID'].max()})")
print(f"Countries : {sorted(raw['Country'].unique().tolist())}")
print()
print(raw['Day of Week'].value_counts())

assert raw.shape == (249231, 16)
assert n_drivers == 400 and n_weeks == 32
assert raw.isna().sum().sum() == 0
assert raw['Route ID'].is_monotonic_increasing, 'Routes are expected in date order'
print('\nAll checks passed.')

Routes    : 19,647
Drivers   : 400
Weeks     : 32 (Week ID 0 to 31)
Countries : [0, 1]

Day of Week
Tuesday      53465
Monday       49013
Thursday     47627
Wednesday    47622
Friday       42736
Saturday      6536
Sunday        2232
Name: count, dtype: int64

All checks passed.


## 5. Build one row per route

The raw table has one row per stop visit, but the prediction is made for a whole route, so the stop rows are
grouped into one row per route.

**Target.** The route duration is the time between the first and the last recorded arrival in the route:

$$\text{duration\_min} = \max(\text{Arrived Time}) - \min(\text{Arrived Time})$$

**Features.** To avoid data leakage, the features only use information that the planner already has before the
driver leaves the depot: the planned stop sequence (`IndexP`), the planned leg distances (`DistanceP`), the
customer time windows, the stop types, the country, the weekday and the driver's history. The columns that record
what actually happened (`IndexA`, `DistanceA`, `Arrived Time`) are only used to build the target and the
data-quality flags, never as model inputs.

| Feature | Meaning |
|---|---|
| `n_stops` | Number of customer stop visits |
| `n_depot_visits` | Number of times the plan passes through the depot |
| `n_pickups` | Number of pickup stops |
| `n_addresses` | Number of different customer addresses |
| `planned_km` | Total planned driving distance |
| `max_leg_km` | Longest single planned leg |
| `first_leg_km` | Planned distance from the depot to the first customer |
| `earliest_start` | Earliest opening time of any customer time window |
| `latest_end` | Latest closing time of any customer time window |
| `mean_window` | Average customer time-window width |
| `min_window` | Narrowest customer time window |
| `country` | Country of operation (0 or 1) |
| `weekday` | Day of the week of the first planned stop |
| `driver_prior_routes` | Number of routes the driver completed in earlier rows of the data |

Two quality flags are also created for the cleaning step: whether the arrival times increase along the actual
driving order, and whether the actual route starts at the depot.

In [5]:
stops = raw.copy()
stops['is_stop'] = (stops['Depot'] == 0).astype(int)
stops['is_pickup'] = ((stops['Depot'] == 0) & (stops['Delivery'] == 0)).astype(int)
stops['window_width'] = stops['Latest Time'] - stops['Earliest Time']

planned = stops.sort_values(['Route ID', 'IndexP'])
actual = stops.sort_values(['Route ID', 'IndexA'])
by_plan = planned.groupby('Route ID')
by_actual = actual.groupby('Route ID')
by_customer = planned[planned['is_stop'] == 1].groupby('Route ID')

routes = pd.DataFrame({
    'week_id': by_plan['Week ID'].first(),
    'driver_id': by_plan['Driver ID'].first(),
    'country': by_plan['Country'].first(),
    'weekday': by_plan['Day of Week'].first(),
    'n_stops': by_plan['is_stop'].sum(),
    'n_depot_visits': by_plan['Depot'].sum(),
    'n_pickups': by_plan['is_pickup'].sum(),
    'n_addresses': by_customer['Address ID'].nunique(),
    'planned_km': by_plan['DistanceP'].sum(),
    'max_leg_km': by_plan['DistanceP'].max(),
    'first_leg_km': by_customer['DistanceP'].first(),
    'earliest_start': by_customer['Earliest Time'].min(),
    'latest_end': by_customer['Latest Time'].max(),
    'mean_window': by_customer['window_width'].mean(),
    'min_window': by_customer['window_width'].min(),
    'duration_min': by_actual['Arrived Time'].max() - by_actual['Arrived Time'].min(),
    'flag_in_order': by_actual['Arrived Time'].apply(lambda t: t.is_monotonic_increasing),
    'flag_depot_first': by_actual['Depot'].first() == 1,
}).reset_index()

routes['driver_prior_routes'] = routes.groupby('driver_id').cumcount()

print(f'Stop rows : {len(stops):,}')
print(f'Routes    : {len(routes):,}')
print(f'Columns   : {routes.shape[1]}')
display(routes.head())

Stop rows : 249,231
Routes    : 19,647
Columns   : 20


,Route ID,week_id,driver_id,country,weekday,n_stops,n_depot_visits,n_pickups,n_addresses,planned_km,max_leg_km,first_leg_km,earliest_start,latest_end,mean_window,min_window,duration_min,flag_in_order,flag_depot_first,driver_prior_routes
0,0,0,0,1,Monday,5,2,0,3.00,49.47,16.33,16.33,60.00,540.00,372.00,240.00,331.28,True,True,0
1,1,0,1,1,Monday,6,1,0,6.00,33.27,16.77,16.77,120.00,540.00,360.00,180.00,306.53,True,True,0
2,2,0,2,1,Monday,6,1,0,6.00,12.12,8.22,8.22,120.00,600.00,320.00,180.00,362.99,True,True,0
3,3,0,3,1,Monday,9,1,0,8.00,19.04,14.69,14.69,120.00,600.00,306.67,120.00,254.09,True,True,0
4,4,0,4,1,Monday,7,1,0,7.00,20.63,10.36,10.36,240.00,780.00,334.29,240.00,160.75,True,True,0


### 5.1 Check the route table

The assertions confirm that no stop row was lost or double-counted when the rows were grouped, and that the
routes are still in date order. The summary of the target and the flags shows how much of the data has quality
problems before any cleaning.

In [6]:
assert routes['Route ID'].is_unique
assert len(routes) == raw['Route ID'].nunique()
assert (routes['n_stops'] + routes['n_depot_visits']).sum() == len(raw)
assert routes['Route ID'].is_monotonic_increasing and routes['week_id'].is_monotonic_increasing

print('Route duration (minutes), before cleaning:')
display(routes['duration_min'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_frame().T)

print(f"Arrival times in driving order : {routes['flag_in_order'].mean():.1%}")
print(f"Actual route starts at depot   : {routes['flag_depot_first'].mean():.1%}")
print(f"Routes with no customer stops  : {(routes['n_stops'] == 0).sum()}")
print(f"Routes with zero duration      : {(routes['duration_min'] == 0).sum()}")
print(f"Routes longer than 24 hours    : {(routes['duration_min'] > 1440).sum()}")

Route duration (minutes), before cleaning:


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
duration_min,"19,647.00",451.77,"1,351.41",0.00,0.00,5.76,267.28,354.16,444.05,624.80,"2,787.85","57,552.85"


Arrival times in driving order : 92.3%
Actual route starts at depot   : 94.7%
Routes with no customer stops  : 30
Routes with zero duration      : 738
Routes longer than 24 hours    : 390


## 6. Clean the route table

Section 5.1 showed that some routes contain recording errors: arrival times out of order, routes that do not
start at the depot, durations of zero and durations of several days. These routes do not describe a real delivery
shift, so they are removed. Each rule is based on a physical or legal limit rather than on how unusual a value
looks, so genuine long or short routes are kept.

| Rule | A route is kept only if… | Reason |
|---|---|---|
| R1 | it has at least one customer stop | A route that only visits the depot is not a delivery route. |
| R2 | its arrival times increase along the actual driving order | A driver cannot arrive at a later stop before an earlier one, so the timestamps are corrupted. |
| R3 | its actual driving order starts at the depot | The duration is measured from the depot departure; without it the start time is unknown. |
| R4 | all customer time windows open and close within one day (0–1,440 minutes) | A same-day delivery window cannot close days later; values such as 12.8 million minutes are recording errors. |
| R5 | its duration is at most 780 minutes (13 hours) | Workers in Norway and Denmark must have at least 11 consecutive hours of rest in every 24 hours (Directive 2003/88/EC), so one shift cannot span more than 13 hours. |
| R6 | its implied average speed (planned km ÷ duration) is at most 130 km/h | 130 km/h is the highest legal speed limit in either country, so a faster route is physically impossible. |
| R7 | it takes at least one minute per customer stop | Handing over or collecting a parcel takes time, so a shorter route cannot have been driven. |

The rules are applied in order, and the log records how many routes each rule removes. The same rules are applied
to every week of data before the split, because they describe whether a record is valid, not how the model is
trained.

In [7]:
speed_kmh = routes['planned_km'] / (routes['duration_min'] / 60)

rules = {
    'R1 has customer stops': routes['n_stops'] > 0,
    'R2 arrivals in driving order': routes['flag_in_order'],
    'R3 starts at the depot': routes['flag_depot_first'],
    'R4 time windows within one day': (routes['earliest_start'] <= 1440) & (routes['latest_end'] <= 1440),
    'R5 duration at most 13 hours': routes['duration_min'] <= 780,
    'R6 average speed at most 130 km/h': (routes['duration_min'] > 0) & (speed_kmh <= 130),
    'R7 at least 1 minute per stop': routes['duration_min'] >= routes['n_stops'],
}

keep = pd.Series(True, index=routes.index)
log = []
for name, passed in rules.items():
    passed = passed.fillna(False).astype(bool)
    removed = int((keep & ~passed).sum())
    keep &= passed
    log.append({'rule': name, 'failed (any order)': int((~passed).sum()),
                'removed at this step': removed, 'routes left': int(keep.sum())})

cleaning_log = pd.DataFrame(log)
cleaning_log.to_csv(os.path.join(RESULTS_DIR, 'cleaning_log.csv'), index=False)
display(cleaning_log)

routes_clean = routes.loc[keep].drop(columns=['flag_in_order', 'flag_depot_first']).reset_index(drop=True)
print(f'Routes kept    : {len(routes_clean):,} of {len(routes):,} ({len(routes_clean) / len(routes):.1%})')
print(f'Routes removed : {len(routes) - len(routes_clean):,}')

,rule,failed (any order),removed at this step,routes left
0,R1 has customer stops,30,30,19617
1,R2 arrivals in driving order,1513,1509,18108
2,R3 starts at the depot,1045,1020,17088
3,R4 time windows within one day,351,217,16871
4,R5 duration at most 13 hours,604,416,16455
5,R6 average speed at most 130 km/h,1002,405,16050
6,R7 at least 1 minute per stop,959,2,16048


Routes kept    : 16,048 of 19,647 (81.7%)
Routes removed : 3,599


### 6.1 Check the cleaned table

The assertions confirm that every rule holds in the cleaned table, that no values are missing, and that every
week still has enough routes for a chronological split. The summary shows the target after cleaning.

In [8]:
assert routes_clean.isna().sum().sum() == 0
assert routes_clean['duration_min'].between(1, 780).all()
assert (routes_clean['duration_min'] >= routes_clean['n_stops']).all()
assert routes_clean['latest_end'].le(1440).all() and routes_clean['earliest_start'].le(1440).all()
assert routes_clean['Route ID'].is_monotonic_increasing

print('Route duration (minutes), after cleaning:')
display(routes_clean['duration_min'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_frame().T)

per_week = routes_clean.groupby('week_id').size()
print(f'Routes per week: min {per_week.min()}, median {per_week.median():.0f}, max {per_week.max()}')
assert per_week.min() >= 300

routes_clean.to_pickle(os.path.join(DATA_DIR, 'routes_clean.pkl'))
print('Saved:', os.path.join(DATA_DIR, 'routes_clean.pkl'))

Route duration (minutes), after cleaning:


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
duration_min,"16,048.00",358.26,120.11,2.16,94.70,167.62,277.04,352.98,434.49,568.16,664.82,776.73


Routes per week: min 350, median 491, max 617
Saved: /content/gdrive/MyDrive/IntroToAI/AT1_II/data/routes_clean.pkl
